<a href="https://colab.research.google.com/github/terry0809000/Artificial-Intelligence-KCL/blob/main/7PAVAIHA_2026_Practical_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 3: Particle Swarm Optimisation for Feature Selection

## IMPORTANT

Please save your own copy of this workbook and work off that. Click on `File` and select `Save a copy in Drive`

Then, click on `Runtime` above and if highlighted, click on `Restart session`

Finally, click on `edit` above and click on `Clear all outputs`

#### Compute Power

This practical will only require CPU runtime.

Step 1: Open Runtime Settings
*   Click Runtime in the top menu.
*   Click Change runtime type.

Step 2: Select CPU, then click on Save

<hr style="border:1px solid black"> </hr>

## Introduction

In today's lab, we will learn how to perform feature selection using **Particle Swarm Optimisation (PSO)**, utilising the [pyswarms](https://pyswarms.readthedocs.io/en/latest/index.html) toolkit.

**Question:** Why might we want to perform feature selection?
- What are some advantages and disadvantages of using it in machine learning pipelines?

## Step 1: Install and importing relevant modules

In [1]:
pip install pyswarms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.1 MB/s eta 0:00:00


In [2]:
# Import modules
import numpy as np
import time

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Import PySwarms
import pyswarms as ps

## Step 2: Create dataset with redundant features

Use `sklearn`'s __[make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)__ function to create a dataset with a large number of redundant features.

**Task:** Add comments to the code explaining the core steps in detail.
- Focus on what each parameter and step is doing, not just what the code says.
- You can use an AI assistant to help support your understanding.

In [3]:
# Define the number of features for each type
n_features = 200
n_informative = 20
n_redundant = 100
n_repeated = 50
n_useless = 30

# Create Labels
informative_labels = [f'Informative {ii}' for ii in range(1, n_informative + 1)]
redundant_labels = [f'Redundant {ii}' for ii in range(n_informative + 1, n_informative + n_redundant + 1)]
repeated_labels = [f'Repeated {ii}' for ii in range(n_informative + n_redundant+ 1, n_informative + n_redundant + n_repeated + 1)]
useless_labels = [f'Useless {ii}' for ii in range(n_informative + n_redundant + n_repeated + 1, n_features + 1)]
labels = informative_labels + redundant_labels + repeated_labels + useless_labels

# Get data
X, y = make_classification(n_samples = 5000, n_features = n_features,
                           n_informative = n_informative,
                           n_redundant = n_redundant , n_repeated = n_repeated,
                           n_clusters_per_class = 2, class_sep = 0.5, flip_y = 0.05,
                           random_state = 42, shuffle = False)


We now split the data into training and test sets.

**Question** Why do we need to do this?

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

The role of the code in the next cell is to standardise the data.

**Question** What does standardisation do to the data, and why might this be important for feature selection?

In [ ]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Step 3: Set-up the objective function

We’ll be using the optimizer `pyswarms.discrete.BinaryPSO` to perform feature subset selection.

For a Binary PSO, the position of the particles are expressed in two terms: 1 or 0 (or on and off).

Mathematically, this is defined as:

$x=[x_{1},x_{2},x_{3},…,x_{d}]$

where:

$x_{i}\in {0,1}$

The objective function we will be using is taken from this __[paper](https://www.sciencedirect.com/science/article/abs/pii/S1568494613001361)__, and is define as:

$f(X)=\alpha(1−P)+(1−\alpha)(1− \dfrac{N_{f}}{N_{t}})$

Where
- $\alpha$ is a hyperparameter tradeoffs the performance of classifier
- $P$, with the ratio of the size of the feature subset
- $N_{f}$ with respect to the total number of features $N_{t}$

<hr style="border:1px solid black"> </hr>

#### Note

The twin goals of feature selection are to maximise model performance while minimising the number of selected features.

The term $\alpha(1 - P)$ represents the classification error (i.e. how well the model performs), while $(1 - \alpha)\left(1 - \dfrac{N_f}{N_t}\right)$ represents the proportion of selected features relative to the total number available.

By adjusting $\alpha$, we are effectively trading off between classification performance and selecting fewer features.

However, we need to be careful. If $\alpha$ is weighted too heavily towards performance, we may end up with more features than training instances.

**Question**
What is this phenomenon called, and what effect might it have on classifier performance?

<hr style="border:1px solid black"> </hr>

We now need need to define a function which calcualtes the objective function per particle.

Our function `f_per_particle` a binary mask and $\alpha$ and returns the object loss score for a *single* particle

In [ ]:
# Create an instance of the classifier
classifier = RandomForestClassifier(max_depth=2,n_estimators=10)

# Define objective function
def f_per_particle(m, alpha):

    X_train_subset, X_test_subset, y_train_subset, y_test_subset = train_test_split(X_train, y_train, test_size=0.33)

    # Get the subset of the features from the binary mask
    if np.count_nonzero(m) == 0:
        X_train_masked = X_train_subset
        X_test_masked = X_test_subset
    else:
        X_train_masked = X_train_subset[:,m==1]
        X_test_masked = X_test_subset[:,m==1]

    # Perform classification and store performance in P
    classifier.fit(X_train_masked, y_train_subset)
    P = (classifier.predict(X_test_masked) == y_test_subset).mean()

    # Compute for the objective function
    j = (alpha * (1.0 - P)
        + (1.0 - alpha) * (1 - (X_train_masked.shape[1] /X_train_subset.shape[1])))

    return j

Next we define the higher level objective function to be evaluated in pyswarms built in optimiser.

`f` returns the object loss score for *all* particles

In [ ]:
def f(M, alpha=0.5):
    """Higher-level method to do classification in the
    whole swarm.

    Inputs
    ------
    M: numpy.ndarray of shape (n_particles, dimensions)
        The swarm that will perform the search

    Returns
    -------
    numpy.ndarray of shape (n_particles, )
        The computed loss for each particle
    """

    n_particles = M.shape[0]

    j = [f_per_particle(M[i], alpha) for i in range(n_particles)]

    return np.array(j)

## Step 4: Set-up the optimisation parameters and run the selections algorithm

In the `options` dictionary below:
- `k`represents the neighbors to be considered when calculating the best known position of the swarm
- `p`represents a distance metric used in the optimisation algorithm.

**Question** What are the purpose of the other parameters?

### Task: Track Runtime

Update the code below so that you can measure how long the optimiser takes to run and report this result.

In [ ]:
# Initialize swarm, arbitrary
options = {'c1': 1.8, 'c2': 0.5, 'w':0.5, 'k': 20, 'p':2}

# Call instance of PSO
dimensions = X.shape[1] # dimensions should be the number of features

optimizer = ps.discrete.BinaryPSO(n_particles=25, dimensions=dimensions, options=options)

# Perform optimization
cost, pos = optimizer.optimize(f, iters=10, verbose=2)
selected = np.count_nonzero(pos)
print("Optimiser retained " + str(selected)   + " features")

NameError: name 'X' is not defined

## Step 5: Evalute the performamce of the selected features

We evaluate the effectiveness of the selected features, we can use two Random Forest Classifiers, one trained on the full feature set, the other trained on the selected feature sets, and compare the outputs

In [ ]:
c1 = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)
c1.fit(X_train, y_train)

full_performance = (c1.predict(X_test) == y_test).mean()
print('Full Feature set performance: %.3f' % (full_performance))

c2 = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)
c2.fit(X_train[:,pos==1], y_train)

subset_performance = (c2.predict(X_test[:,pos==1]) == y_test).mean()
print('Subset Feature set performance: %.3f' % (subset_performance))

## Exercise 1

### Exploration Task

Investigate how changing key parameters affects both the performance and running time of the optimiser.

Parameters to experiment with:
- `alpha`
- Swarm options
- Number of particles
- Number of optimiser iterations

**Systematic approach:**
- Change **one parameter at a time** while keeping others constant  
- Record the effect on performance and runtime  
- Compare your results across different settings  

---

**Parameter Ranges**

To keep runtime reasonable, use the following ranges:

- `n_particles`: **10, 20, 50**
- `iters`: **10, 20, 30**
- `alpha`: **0.3, 0.6, 0.9**
- `c1`, `c2`, `w`: *small adjustments only* (±0.25 from original values)

---

### Reflection:
Did you observe any trade-off between performance and computational cost?

## Exercise #2

### Feature Selection on Breast Cancer Dataset

In this exercise, you will use the *Breast Cancer Wisconsin Diagnostic Dataset*.

- Features are computed from digitised images of fine needle aspirates (FNA) of breast masses  
- Each sample is labelled as:
  - **Malignant (0)**
  - **Benign (1)**  
- The dataset can be imported directly from `sklearn`

---

#### Part 1 — Simple Filter Method

Apply a basic **filter-based feature selection method** and evaluate its impact.

**Tasks:**
1. Implement and apply a filter method

2. Train a **Random Forest Classifier** on:
   - The **full feature set**
   - The **reduced feature set**

3. Compare:
   - Model accuracy
   - Number of selected features

---

#### Part 2 — Particle Swarm Optimisation (PSO)

Re-implement **Particle Swarm Optimisation (PSO)** for feature selection using the full dataset.

**Tasks:**
1. Apply PSO to select an optimal subset of features  
2. Use a **systematic approach** to tune hyperparameters (e.g. grid search or controlled experimentation):
   - Number of particles  
   - Number of iterations  
   - Cognitive and social coefficients  

3. Train the same Random Forest model using:
   - Features selected by PSO  

4. Record:
   - Model performance (e.g. accuracy)
   - Number of selected features  

---

#### Comparison Table

Fill in the table below:

| Method             | Accuracy| # Features| Notes|
|--------------------|---------|-----------|------|
| Full Dataset       |         | 30        |      |
| Filter Method      |         |           |      |
| PSO Selection      |         |           |      |

---

#### Reflection

Consider the following questions:

**Does feature selection improve performance on this dataset?**  
   - Did accuracy increase, decrease, or stay similar?

**Did one method performed better (Filter vs PSO)?**  
   - Provide evidence from your results  
   - Why do you think this was the case?  

**Trade-offs:**  
   - Which method is more computationally expensive?  
   - Which method is easier to interpret?

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
print(data.DESCR)

X, y = data.data, data.target